# LLAMBO, one knob at a time

**Choose a polynomial degree → fit a curve → measure its error → choose again.**

This small Python implementation follows [Liu et al., *Large Language Models to Enhance
Bayesian Optimization* (ICLR 2024)](https://arxiv.org/pdf/2402.03921).
An LLM proposes settings and predicts their scores; Python runs every real experiment.

The only knob in this demo is **polynomial degree, from 1 to 12**. We fit synthetic noisy
samples, then minimize mean squared error (MSE) on separate validation samples. Lower is better.

Run from top to bottom using **GNLSE (Python)**. The live run makes **9 Codex
calls and 5 model fits** with the defaults below. It uses your existing `codex login` and model
selection. There is no API key in this notebook. Only the run cell contacts the LLM.

On another machine: install `requirements.txt`, install the Codex CLI, and run `codex login`.

## 1. Setup

Three starting experiments, followed by two LLM-guided experiments. Each candidate gets
three predictions with differently shuffled observations. The standard deviation of those
predictions is a rough uncertainty estimate.

In [ ]:
from datetime import datetime
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import norm
from IPython.display import display

workspace = next(folder for folder in (Path.cwd(), *Path.cwd().parents) if (folder / "notebook_helpers" / "run_library.py").is_file())
sys.path.insert(0, str(workspace / "notebook_helpers" / "llambo_mvp"))
# The client records each model call and writes the experiment settings.
from llm_client import connect_llm, write_json

N_INITIAL = 3
N_STEPS = 2
N_CANDIDATES = 4
N_PREDICTIONS = 3
SEED = 7

## 2. The demo problem

The training points contain noise. Low-degree curves can miss the shape; very flexible
curves can follow the noise. We compare degrees on a fixed validation set so changing
random data cannot change the ranking. A small, fixed ridge penalty stabilizes the fit.

**To use another experiment later, replace `problem` and `evaluate(parameters)`.**
The optimizer only needs a parameter dictionary and a finite scalar score to minimize.

In [ ]:
data_rng = np.random.default_rng(42)
x_train = np.sort(data_rng.uniform(-1, 1, 24))
x_valid = np.linspace(-1, 1, 80)
y_train = np.sin(3 * x_train) + data_rng.normal(0, 0.18, len(x_train))
y_valid = np.sin(3 * x_valid) + data_rng.normal(0, 0.18, len(x_valid))

problem = {
    "description": (
        "Choose a polynomial degree for ridge regression on 24 noisy training samples. "
        "There is one input in [-1, 1] and a smooth nonlinear target. "
        "The fixed ridge penalty is 0.0001. Score on 80 fixed validation samples. "
        "Higher degrees fit more complex curves but may overfit."
    ),
    "metric": "validation mean squared error; minimize",
    "parameters": {"degree": {"type": "integer", "minimum": 1, "maximum": 12}},
}


# Bind the training and validation samples used in each trial.
from functools import partial
from polynomial_demo import evaluate as validation_error, fit_curve as fit_polynomial

fit_curve = partial(fit_polynomial, x_train=x_train, y_train=y_train)
evaluate = partial(validation_error, x_train=x_train, y_train=y_train, x_valid=x_valid, y_valid=y_valid)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.4), constrained_layout=True)
ax.scatter(x_train, y_train, color="#226b9b", label="Training samples")
ax.scatter(x_valid, y_valid, s=12, color="#b57938", alpha=0.5, label="Validation samples")
ax.set(xlabel="Input x", ylabel="Target y", title="One knob: how complex should the curve be?")
ax.legend(frameon=False)
plt.show()

## 3. Ask for starting points

The LLM sees the problem description but no results yet (paper §4).
The small JSON helpers in `llm_client.py` enforce parameter names, types, ranges, and uniqueness.

In [ ]:
# Suggest distinct initial settings from the problem description.
from polynomial_demo import suggest_starting_points

## 4. Ask for candidates

After observing scores, request settings targeting
$y_{target} = y_{best} - \alpha (y_{worst} - y_{best})$ (paper §6).
Here $\alpha=0.01$: aim a little beyond the best measured result.

In [ ]:
# Propose new settings aiming slightly below the best observed score.
from polynomial_demo import suggest_candidates

## 5. Predict each candidate's score

Repeat score prediction with shuffled examples, then calculate a mean and standard deviation
(paper §5.1). Candidate IDs keep predictions matched to their settings.

In [ ]:
# Estimate candidate means and uncertainty from repeated LLM predictions.
from polynomial_demo import predict_scores

## 6. Choose by expected improvement

A candidate can be attractive because its predicted error is low or because its uncertainty
leaves room for improvement. Expected improvement (EI) combines these two considerations.
We choose the candidate with the **largest EI**, then measure it.

In [ ]:
# Expected improvement for minimization under a Gaussian approximation.
from polynomial_demo import expected_improvement

## 7. Put the steps together

The history contains **measured scores only**. Candidate predictions are saved separately.
Every completed experiment is saved immediately. Rerunning the run cell starts a fresh run.

In [ ]:
# Warm-start, propose, predict, select, measure, and save.
from polynomial_demo import run_optimization

## 8. Run

This is the only cell that calls Codex. It inherits your current model configuration.
Prompts, replies, predictions, and measured history are saved under the project's `output/`
folder. The call budget is explicit; failures stop visibly and keep earlier measurements.

In [ ]:
run_name = datetime.now().astimezone().strftime("%Y-%m-%d_llambo_mvp_%H%M%S_%f")
run_dir = workspace / "output" / "organized" / "runs" / "Polynomial degree (synthetic)" / "LLM-assisted Bayesian optimization" / run_name
max_calls = 1 + N_STEPS * (1 + N_PREDICTIONS)
ask_llm = connect_llm(run_dir / "llm_calls", max_calls=max_calls)
write_json(run_dir / "settings.json", {
    "problem": problem, "n_initial": N_INITIAL, "n_steps": N_STEPS,
    "n_candidates": N_CANDIDATES, "n_predictions": N_PREDICTIONS,
    "seed": SEED, "data_seed": 42, "alpha": 0.01,
    "max_llm_calls": max_calls, "backend": "Codex CLI (inherited model)",
})

results = run_optimization(
    problem, evaluate, ask_llm, run_dir,
    n_initial=N_INITIAL, n_steps=N_STEPS,
    n_candidates=N_CANDIDATES, repeats=N_PREDICTIONS, seed=SEED,
)
print(f"\nSaved to: {run_dir}")
display(results.round({"score": 5}))

## 9. See the result

The best measured setting is the recommendation. The validation set guides the search;
its score is not an independent test-set estimate. This tiny run demonstrates the workflow,
not a performance comparison with other optimizers.

In [ ]:
best = results.loc[results["score"].idxmin()]
best_degree = int(best["degree"])
print(f"Best measured degree: {best_degree}; validation MSE: {best['score']:.5f}")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), constrained_layout=True)
axes[0].plot(results["trial"], results["score"], "o", color="#7b98ad", label="Measured error")
axes[0].step(results["trial"], results["score"].cummin(), where="post",
             color="#226b9b", label="Best so far")
axes[0].set(xlabel="Experiment", ylabel="Validation MSE", title="Optimization history")
axes[0].set_xticks(results["trial"])
axes[0].legend(frameon=False)
axes[1].scatter(x_train, y_train, s=24, color="#7b98ad", label="Training samples")
axes[1].plot(x_valid, np.polynomial.polynomial.polyval(x_valid, fit_curve(best_degree)),
             color="#226b9b", label=f"Selected fit: degree {best_degree}")
axes[1].set(xlabel="Input x", ylabel="Prediction", title="Best measured curve")
axes[1].legend(frameon=False)
fig.savefig(run_dir / "result.png", dpi=180)
plt.show()

## 10. Plug in your own experiment

Keep the optimization functions. Change the parameter definitions and return your measured
objective from `evaluate`. Numeric parameters can use `"integer"` or `"number"` types.
For example, the interface for a one-power optical search would be:

```python
problem = {
    "description": "Describe the fixed fiber, pulse, modes, and numerical settings here.",
    "metric": "D_IO; minimize",
    "parameters": {
        "power_kW": {"type": "number", "minimum": 50, "maximum": 150},
    },
}

def evaluate(parameters):
    result = your_existing_simulator(power_kW=parameters["power_kW"])
    return float(result["D_IO"])
```

This is an interface example, not a connected optical run. Set physically justified bounds
and handle brightness or other feasibility requirements in your evaluator before using it.
The present MVP handles one finite scalar objective within fixed numeric bounds.
The two plot cells are specific to the curve-fitting demo; replace them for another problem.

### How close is this to the paper?

Implemented: warm-starting, target-conditioned proposals, shuffled repeated score predictions,
and expected improvement. For simplicity, candidates and their predictions are batched in each
call; the LLM is your configured Codex model. This is a small adaptation, not a reproduction
of the paper's GPT-3.5 benchmarks. Prediction spread is heuristic, not calibrated uncertainty.

Sources: [paper, §§4–7](https://arxiv.org/pdf/2402.03921) ·
[official structured Codex-call example](https://developers.openai.com/cookbook/examples/codex/build_iterative_repair_loops_with_codex)